### Classification vs. Regression

I considered treating stress as a continuous variable, but since it comes from ordered Likert-scale categories, I chose classification instead. My goal is to identify which indicators predict different stress levels, and classification makes this easier to interpret and analyze using feature importance.

Therefore, I will only be using **classification**

### Evaluation Metrics

- The primary metric used will be **F1**
- Accuracy
- Precision and Recall: Break down what kind of mistakes the model makes
  - Recall sees of all the actual High-stress days, how many did the model catch?
  - Precision sees of all the days it labeled High-stress, how many really were there (i.e. false alarms) ?
- Confusion matrix: Shows which classes get confused


### Classification Models

As I said, I will only be using classification models. Based on data splitting and validation, I will **end up with one trained model per phase**.


**Classification Models**:
- Logistic Regression
- Random Forest
- XGBoost

First I want to know how much data I have to decide on the hyperparameters



In [ ]:
for phase in merged['phase'].unique():
    subset = merged[merged['phase'] == phase]
    print(f"{phase}: {len(subset)} rows, {subset['id'].nunique()} participants, {len(feature_cols)} features")

# different row count because the phases have different lengths for each person and different amounts of data collected

Follicular: 839 rows, 42 participants, 20 features
Fertility: 732 rows, 42 participants, 20 features
Luteal: 1115 rows, 42 participants, 20 features
Menstrual: 633 rows, 42 participants, 20 features


I have a pretty good amount of data so can support some model complexity with some deeper trees and moderate regularization.

Therefore, I will use the hyperparameters listed below.

**Hyperparameter and Regularization**

**Logistic Regression**

*Why* : It's fast and interpretable. Bias-variance: it's the higher-bias, lower-variance end so could underfit. That's why I will use it a baseline model. I will also apply regularization with the strength tuned via cross-validation to balance bias and variance.

*Regularization* : L2. The C (or lambda as in CSC311) parameter controls it (smaller C = stronger regularization = simpler model). Controls variance/overfitting).

*Hyperparameters to tune* : C (regularization strength): [0.01, 0.1, 1, 10]

*Optimization* : Handled automatically by sklearn


**Random Forest**

*Why* : Handles non-linearity and provides built-in feature importance. It's also a **bagging** ensemble **as we learned in class**, reducing variance by averaging many decision trees. **Reduces overfitting. **

*Regularization* : Controlled by hyperparameters. **Note that shallower trees reduce overfitting.**

*Hyperparameters to tune* : n_estimators (number of trees), max_depth (tree depth — key overfitting control), min_samples_leaf (minimum samples per leaf).

n_estimators:     [100, 200, 300]

max_depth:        [4, 6, 8, 12]

min_samples_leaf: [2, 5, 10]

**XGBoost**

*Why* : XGBoost is built on the principles of ensemble learning and gradient boosting. XGBoost builds trees one after another keeping in mind the mistakes that were made. It uses **boosting** as opposed to bagging like Random Forest, primarily reducing bias instead so helping **fix underfitting**
Typically the strongest performer on tabular data and handles imbalance well.

*Regularization* : L2

*Hyperparameters to tune* : learning_rate, max_depth, n_estimators, subsample.

learning_rate: [0.05, 0.1]

max_depth:     [3, 4, 6]

n_estimators:  [100, 200, 300]

subsample:     [0.8, 1.0]

**Additional**

**Tuning method**: **Random Search** over hyperparameters inside StratifiedGroupKFold cross-validation, while optimizing macro F1 (random search finds good regions faster than grid search over a large space).

**Class weights**: optionally class_weight='balanced' on Logistic Regression and Random Forest, given the mild class imbalance

### How I'll choose a model

See if models are within one standard deviation of each other OR just whichever gives more interpretable data

**Using Logistic Regression (with random search)**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

feature_cols = [
    'mean_temp', 'max_temp', 'min_temp', 'std_temp',
    'mean_rmssd', 'max_rmssd', 'min_rmssd', 'std_rmssd',
    'mean_high_frequency', 'max_high_frequency', 'min_high_frequency', 'std_high_frequency',
    'mean_bpm', 'max_bpm', 'min_bpm',
    'mean_vo2', 'max_vo2', 'min_vo2',
]
target_col = 'stress_encoded'

# pipeline = scaling + model (preventing data leakage during cross-validation)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(penalty='l2', class_weight='balanced', max_iter=1000)),
])

# hyperparameter
param_dist = {
    'clf__C': [0.01, 0.1, 1, 10],
}

results = {}

# this for loop is for stratificaiton where I build a separate model for each
# cycle phase
for phase in merged['phase'].unique():
    subset = merged[merged['phase'] == phase]
    X = subset[feature_cols]
    y = subset[target_col]
    groups = subset['id']

    sgkf = StratifiedGroupKFold(n_splits=5)

    search = RandomizedSearchCV(
        pipe,
        param_dist,
        n_iter=4, # trying each C value
        cv=sgkf,
        scoring='f1_macro',
        random_state=42,
        n_jobs=-1,
    )
    search.fit(X, y, groups=groups)
    # training many models (each C × each fold) and finding the best one

    results[phase] = search
    print(f"{phase}: best C = {search.best_params_['clf__C']}, "
          f"macro F1 = {search.best_score_:.3f}")

Follicular: best C = 0.1, macro F1 = 0.311
Fertility: best C = 0.1, macro F1 = 0.295
Luteal: best C = 0.1, macro F1 = 0.312
Menstrual: best C = 10, macro F1 = 0.342


###Feature Importance &  Interpretability

**Feature Importance**

Remember I will compute feature importance separately for each phase's model, then compare. **The idea is to use feature importance to answer my research question**. For example "In the luteal phase, temperature ranked highest". This is useful to answer the question.

I will get feature importance through:
- Built-in importance automatically produced by the **tree models** during training.
- For the **logistic regression**, the size of each feature's coefficient tells me its influence (after scaling, so they're comparable). Bigger absolute coefficient = stronger effect on the prediction.

I could also use **permutation importance** for **pruning** (still deciding)


**Interpretability**

To interpret the data I will use **SHAP**. It is a method for explaining why a model made a particular prediction and which features were most important which is perfect. It looks at how much a feature contributed to making this prediction. Importantly, it includes direction, e.g. high temperature pushes toward high stress. Will do a **summary plot per phase**.

### Risks, Limitations, and Next Steps

**Risks and Limitations**
- Even with stratification, phase and stress can interact
- Have only 42 participants
- After aggregation, some noise still remains
- Grouping 6 classes into 3

**Next Steps**
- Find out what's causing the F1-score to be low and make fixes
- Try using ordinal logistic regression as well
- Feature importance and interpretation